# 01 - Data collection and exploration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = '../data/raw/spotify_songs.csv'

## 1. Thu thập dữ liệu

### Chủ đề & Bối cảnh thực tế 
- Chủ đề: Metadata âm nhạc số và các đặc trưng âm thanh định lượng của 32.833 bản ghi bài hát trên nền tảng Spotify, với 23 cột được chia thành 4 nhóm chính theo bảng mô tả của TidyTuesday:
  1. **Định danh bài hát & album (6 cột):** `track_id`, `track_name`, `track_artist`, `track_album_id`, `track_album_name`, `track_album_release_date`.
  2. **Phân loại theo playlist (4 cột):** `playlist_name`, `playlist_id`, `playlist_genre` (6 thể loại chính), `playlist_subgenre` (24 tiểu thể loại).
  3. **Chỉ số mức độ phổ biến (1 cột):** `track_popularity` (thang điểm 0–100 phản ánh độ thịnh hành của bài hát).
  4. **Đặc trưng âm thanh trích xuất từ Spotify API (12 cột):** `danceability`, `energy`, `key`, `loudness`, `mode`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, và `duration_ms`.
- Bối cảnh thực tế: Các nền tảng phát nhạc trực tuyến hiện đại như Spotify ứng dụng lĩnh vực Truy xuất Thông tin Âm nhạc (MIR) để lượng hóa cảm nhận âm nhạc trừu tượng thành các thông số kỹ thuật có cấu trúc. Bộ dữ liệu này phản ánh cách nền tảng dùng đặc trưng tín hiệu âm thanh kết hợp hành vi tuyển chọn playlist để vận hành hệ thống gợi ý và phân tích thị hiếu người nghe theo từng dòng nhạc.

### Nguồn dữ liệu
- **Bản gốc (TidyTuesday):** Dữ liệu từ **Spotify AB**, do **Kaylin Pavlik** thu thập và cộng đồng **R4DS** công bố ngày **21/01/2020** tại [https://github.com/rfordatascience/tidytuesday/tree/master/data/2020/2020-01-21](https://github.com/rfordatascience/tidytuesday/tree/master/data/2020/2020-01-21).
- **Bản tải về (Kaggle):** Được **Joakim Arvidsson** (`joebeachcapital`) đăng tải lại năm **2023** tại [https://www.kaggle.com/datasets/joebeachcapital/30000-spotify-songs](https://www.kaggle.com/datasets/joebeachcapital/30000-spotify-songs).

### Giấy phép sử dụng
- **Kho lưu trữ gốc (TidyTuesday):** Phát hành theo giấy phép **CC0-1.0** (Public Domain), cho phép tự do sao chép, chỉnh sửa và khai thác cho cả mục đích học thuật lẫn thương mại.
- **Bản phân phối (Kaggle):** Áp dụng **Open Database License (ODbL v1.0)**, yêu cầu ghi nhận nguồn đóng góp (R4DS, Kaylin Pavlik, Joakim Arvidsson) và duy trì tính mở nếu tái phân phối cơ sở dữ liệu phái sinh.
- **Ràng buộc bản quyền (Spotify Web API):** Dữ liệu gốc chịu sự điều chỉnh của **Spotify Developer Terms of Service**.

### Phương pháp thu thập & Thiên lệch dữ liệu
- **Cơ chế thu thập:** Truy xuất tự động qua **Spotify Web API** bằng gói thư viện `spotifyr` (R) từ các playlist công khai; tiêu chí chọn playlist được tác giả trình bày chi tiết trong bài blog đính kèm tại README gốc.
-  Lấy mẫu ~5.000 bản ghi cho mỗi thể loại trong **6 thể loại chính** (*EDM, Latin, Pop, R&B, Rap, Rock* — chia thành 24 tiểu thể loại), tổng cộng **32.833 dòng** nhưng chỉ **28.356 bài hát** khác nhau. Ngày phát hành trải dài từ **1957 đến 01/2020**, nhưng cấu trúc playlist và chỉ số `track_popularity` là ảnh chụp tĩnh tại **tháng 01/2020**.
- **Hạn chế & Biases:**
  - **Nhiễu nhãn thể loại (*Label Noise*):** Nhãn `playlist_genre` được kế thừa từ playlist chứa bài hát thay vì thể loại gốc của nghệ sĩ.
  - **Trùng lặp bản ghi (*Multi-playlist Duplication*):** Các bài hát phổ biến xuất hiện ở nhiều playlist gây trùng lặp `track_id` (4.477 dòng lặp lại `track_id`).
  - **Thiên lệch chọn mẫu & văn hóa (*Sampling Bias*):** Tập trung vào 6 dòng nhạc thương mại phổ biến trên Spotify khiến dữ liệu chủ yếu là nhạc Âu - Mỹ (US-UK) và loại trừ hoàn toàn các dòng nhạc ngoài danh mục và tiếng nước ngoài. (*Jazz, Classical, Folk...*).
  - **Thiên lệch thời gian (*Recency Bias*):** Điểm `track_popularity` (0–100) ưu tiên lượt phát gần thời điểm thu thập (01/2020), có thể làm giảm đáng kể điểm số của các bản thu thập niên 1950–1990 (sẽ kiểm tra ở phần khám phá)
  - **Khuyết dữ liệu & Sai lệch định dạng:** Tồn tại 5 bản ghi khuyết (`NA`) ở thông tin định danh và sự không đồng nhất định dạng thời gian ở `track_album_release_date` (`YYYY`, `YYYY-MM-DD` và `YYYY-MM`).
### Lý do chọn bộ dữ liệu
- **Cân bằng giữa quy mô và chiều sâu đặc trưng:** Với 32.833 quan sát cùng 23 thuộc tính kết hợp cả biến phân loại phân cấp (`playlist_genre`, `playlist_subgenre`), biến thời gian, biến mục tiêu (`track_popularity`) và 12 đặc trưng âm thanh liên tục, bộ dữ liệu đủ lớn và đa dạng để thực hiện trọn vẹn quy trình từ EDA, kiểm định thống kê đến xây dựng mô hình học máy.
- **Tính diễn giải trực quan:** Các đặc trưng như độ sôi động (`energy`), độ tích cực cảm xúc (`valence`), nhịp độ (`tempo`) hay độ ồn (`loudness`) dễ liên hệ thực tế, giúp nhóm diễn giải rõ ràng các phát hiện phân tích.
- **Cơ hội rèn luyện tiền xử lý thực tế:** Bộ dữ liệu chứa các vấn đề thực tế như trùng lặp `track_id` do giao thoa playlist, định dạng ngày phát hành không đồng nhất và phân phối lệch mạnh ở các biến như `instrumentalness`, `speechiness` — phù hợp để thể hiện kỹ thuật làm sạch và biến đổi dữ liệu.

In [2]:
df = pd.read_csv(DATA)
df.head()

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


# 2. Khám phá dữ liệu

## 2.1 Overview

In [7]:
df.shape

(32833, 23)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32833 entries, 0 to 32832
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   track_id                  32833 non-null  str    
 1   track_name                32828 non-null  str    
 2   track_artist              32828 non-null  str    
 3   track_popularity          32833 non-null  int64  
 4   track_album_id            32833 non-null  str    
 5   track_album_name          32828 non-null  str    
 6   track_album_release_date  32833 non-null  str    
 7   playlist_name             32833 non-null  str    
 8   playlist_id               32833 non-null  str    
 9   playlist_genre            32833 non-null  str    
 10  playlist_subgenre         32833 non-null  str    
 11  danceability              32833 non-null  float64
 12  energy                    32833 non-null  float64
 13  key                       32833 non-null  int64  
 14  loudness         

In [57]:
df.isna().all(axis=1).sum()

np.int64(0)

### Ý nghĩa từng cột và kiểu dữ liệu mong muốn
| STT | Tên cột | Ý nghĩa | Kiểu dữ liệu mong muốn |
| :--- | :--- | :--- | :--- |
| 0 | `track_id` | ID định danh bài hát. | `str` |
| 1 | `track_name` | Tên bài hát. | `str` |
| 2 | `track_artist` | Tên nghệ sĩ. | `str` |
| 3 | `track_popularity` | Mức độ phổ biến (0–100) của bài hát, giá trị càng cao mức độ phổ biến càng lớn. | `int64` |
| 4 | `track_album_id` | ID định danh của album. | `str` |
| 5 | `track_album_name` | Tên của album. | `str` |
| 6 | `track_album_release_date` | Ngày phát hành của album. | `datetime64` |
| 7 | `playlist_name` | Tên của danh sách phát (playlist). | `str` |
| 8 | `playlist_id` | ID định danh của danh sách phát. | `str` |
| 9 | `playlist_genre` | Thể loại chính của danh sách phát (6 nhóm). | `category` |
| 10 | `playlist_subgenre` | Tiểu thể loại của danh sách phát (24 nhóm). | `category` |
| 11 | `danceability` | Mô tả mức độ phù hợp của bài hát cho việc nhảy (0 -> 1).| `float64` |
| 12 | `energy` | Mức độ cảm nhận về cường độ và sự sôi động của bài hát (0 -> 1).| `float64` |
| 13 | `key` | Cao độ (tông nhạc) tổng thể ước tính của bài hát (Nếu không phát hiện được tông nhạc, giá trị là -1). | `category` |
| 14 | `loudness` | Độ ồn (âm lượng) tổng thể của bài hát tính bằng decibel (dB) (-60 đến 0 dB). | `float64` |
| 15 | `mode` | Điệu thức (âm giai Trưởng hoặc Thứ) của bài hát. Điệu Trưởng (Major) là 1 và điệu Thứ (Minor) là 0. | `category` |
| 16 | `speechiness` | Mức độ xuất hiện của lời nói trong bản thu (0 -> 1). | `float64` |
| 17 | `acousticness` | Độ tin cậy từ 0.0 đến 1.0 xác định xem bài hát có phải là nhạc acoustic hay không | `float64` |
| 18 | `instrumentalness` | Dự đoán xem bài hát là nhạc không lời hay không (các tiếng ngân nga như "ooh", "aah" được tính là nhạc cụ, còn rap hay lời nói được tính là có giọng hát) (0 -> 1). | `float64` |
| 19 | `liveness` | Mức độ hiện diện của khán giả trong bản thu âm. Giá trị càng cao cho thấy xác suất bài hát được biểu diễn trực tiếp (live) càng lớn; giá trị trên 0.8 biểu thị khả năng rất cao đây là bản thu trực tiếp. | `float64` |
| 20 | `valence` | Mô tả sắc thái tích cực về mặt cảm xúc mà bài hát truyền tải. Bài hát có chỉ số cao mang sắc thái tích cực (vui vẻ, tươi sáng, hưng phấn), trong khi chỉ số thấp mang sắc thái tiêu cực (buồn bã, u uất, giận dữ) (0 -> 1) | `float64` |
| 21 | `tempo` | Nhịp độ tổng thể ước tính của bài hát, tính bằng số phách trên mỗi phút (BPM). | `float64` |
| 22 | `duration_ms` | Thời lượng của bài hát tính bằng mili-giây (ms). | `int64` |


### Kiểm tra định dạng dữ liệu các cột ta sẽ đổi kiểu

In [24]:
df['track_album_release_date'].str.len().value_counts()

track_album_release_date
10    30947
4      1855
7        31
Name: count, dtype: int64

In [56]:
sorted(df['key'].unique().tolist())

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

In [55]:
sorted(df['mode'].unique().tolist())

[0, 1]

### Kiểm tra trùng lặp + tìm khóa chính

In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
df.duplicated(subset=['track_id']).sum()

np.int64(4477)

In [13]:
df.duplicated(subset=['track_id', 'playlist_id']).sum()

np.int64(582)

In [14]:
dups = df[df.duplicated(subset=['track_id', 'playlist_id'], keep=False)].sort_values(['track_id', 'playlist_id'])

diff_cols = [col for col in df.columns if dups.groupby(['track_id', 'playlist_id'])[col].nunique().max() > 1]
print("Các cột gây khác biệt:", diff_cols)

dups[['track_id', 'playlist_id'] + diff_cols].head(10)

Các cột gây khác biệt: ['playlist_genre', 'playlist_subgenre']


,track_id,playlist_id,playlist_genre,playlist_subgenre
12979,01R0Xdwje645C6xFCnMRvm,3Ho3iO0iJykgEQNbjB2sic,rock,classic rock
15244,01R0Xdwje645C6xFCnMRvm,3Ho3iO0iJykgEQNbjB2sic,rock,hard rock
27979,037yW9RzsLze4OmBYmcH4G,6KnQDwp0syvhfHOR4lWP7x,edm,electro house
31042,037yW9RzsLze4OmBYmcH4G,6KnQDwp0syvhfHOR4lWP7x,edm,progressive electro house
27251,04wllvXvWOkZS5NugzeS8O,25ButZrVb1Zj1MJioMs09D,edm,electro house
29979,04wllvXvWOkZS5NugzeS8O,25ButZrVb1Zj1MJioMs09D,edm,pop edm
28033,06qKEP3ITnVPmH2ZW1zYvn,6KnQDwp0syvhfHOR4lWP7x,edm,electro house
31097,06qKEP3ITnVPmH2ZW1zYvn,6KnQDwp0syvhfHOR4lWP7x,edm,progressive electro house
18304,070hrICPvpyzXkbqkRC7Ao,4JkkvMpVl4lSioqQjeAL0q,latin,latin pop
19712,070hrICPvpyzXkbqkRC7Ao,4JkkvMpVl4lSioqQjeAL0q,latin,latin hip hop


In [16]:
df.duplicated(subset=['track_id', 'playlist_id','playlist_genre']).sum()

np.int64(323)

In [17]:
df.duplicated(subset=['track_id', 'playlist_id','playlist_subgenre']).sum()

np.int64(0)

### Kiểm tra tính nhất quán của các cột cấp bài hát khi groupby `track_id`

In [58]:
track_level_cols = [
    'track_popularity', 'danceability', 'energy', 'key', 'loudness', 
    'mode', 'speechiness', 'acousticness', 'instrumentalness', 
    'liveness', 'valence', 'tempo', 'duration_ms'
]

nunique_check = df.groupby('track_id')[track_level_cols].nunique()
inconsistent_cols = nunique_check.max()[nunique_check.max() > 1]
print("Các cột bị lệch giá trị giữa các dòng trùng track_id:")
print(inconsistent_cols)

# 2. Đếm số bài hát nằm ở hơn 1 thể loại (playlist_genre)
genres_per_track = df.groupby('track_id')['playlist_genre'].nunique()
multi_genre_count = (genres_per_track > 1).sum()
total_tracks = len(genres_per_track)

print(f"\nTổng số bài độc nhất: {total_tracks}")
print(f"Số bài nằm ở >= 2 thể loại: {multi_genre_count} ({multi_genre_count/total_tracks:.2%})")
print("\nPhân bố số lượng thể loại trên mỗi bài:")
print(genres_per_track.value_counts())

Các cột bị lệch giá trị giữa các dòng trùng track_id:
Series([], dtype: int64)

Tổng số bài độc nhất: 28356
Số bài nằm ở >= 2 thể loại: 1687 (5.95%)

Phân bố số lượng thể loại trên mỗi bài:
playlist_genre
1    26669
2     1422
3      195
4       63
5        7
Name: count, dtype: int64


### Kiểm tra độ phân mảnh (Cardinality) & định dạng giá trị của các cột chuỗi/ID

In [63]:
obj_cols = df.select_dtypes(include='str').columns
cardinality_summary = pd.DataFrame({
    'nunique': df[obj_cols].nunique(),
    'unique_ratio (%)': (df[obj_cols].nunique() / len(df) * 100).round(2),
    'sample_value': [df[col].dropna().iloc[0] for col in obj_cols]
}).sort_values(by='nunique', ascending=False)

display(cardinality_summary)

print("1 playlist_id ứng với tối đa bao nhiêu playlist_name:", df.groupby('playlist_id')['playlist_name'].nunique().max())
print("1 playlist_name ứng với tối đa bao nhiêu playlist_id:", df.groupby('playlist_name')['playlist_id'].nunique().max())
print("1 playlist_subgenre thuộc tối đa bao nhiêu playlist_genre:", df.groupby('playlist_subgenre')['playlist_genre'].nunique().max())


,nunique,unique_ratio (%),sample_value
track_id,28356,86.36,6f807x0ima9a1j3VPbc7VN
track_name,23449,71.42,I Don't Care (with Justin Bieber) - Loud Luxur...
track_album_id,22545,68.67,2oCs0DGTsRO98Gh5ZSl2Cx
track_album_name,19743,60.13,I Don't Care (with Justin Bieber) [Loud Luxury...
track_artist,10692,32.56,Ed Sheeran
track_album_release_date,4530,13.80,2019-06-14
playlist_id,471,1.43,37i9dQZF1DXcZDD7cfEKhW
playlist_name,449,1.37,Pop Remix
playlist_subgenre,24,0.07,dance pop
playlist_genre,6,0.02,pop


1 playlist_id ứng với tối đa bao nhiêu playlist_name: 1
1 playlist_name ứng với tối đa bao nhiêu playlist_id: 5
1 playlist_subgenre thuộc tối đa bao nhiêu playlist_genre: 1


### Kết luận
- Kích thước và đơn vị dòng: 32.833 dòng × 23 cột. Mỗi dòng là một bài hát nằm trong một playlist, ứng với một tiểu thể loại. Khóa chính gồm 3 cột: `track_id` + `playlist_id` + `playlist_subgenre`.
- Trùng lặp:
    - Không có dòng trùng hoàn toàn và không có dòng trống.
    - Có 4.477 dòng lặp `track_id` vì một bài nằm ở nhiều `playlist`. Trong đó **1.687** bài **(5,95%)** thuộc từ 2 thể loại trở lên.
    - Ở bước này ta chưa xóa, việc gộp để sang bước tiền xử lý. Gộp được an toàn vì với cùng `track_id`, `track_popularity` và 12 chỉ số âm thanh luôn giống nhau. Khi gộp phải chọn cách gán thể loại cho các bài nằm ở nhiều thể loại.
- Đổi kiểu:
    - `track_album_release_date` có 3 định dạng (chuỗi dài 10, 7 và 4 ký tự), trong đó 1.855 dòng chỉ có năm. Vì vậy ta chỉ giữ năm phát hành (`release_year`).
    - `key`, `mode`, `playlist_genre` (6 giá trị) và `playlist_subgenre` (24 giá trị) chuyển sang `category`. Cột key không có giá trị -1.
- Cột bỏ:
    - Bỏ `playlist_name` vì 1 tên ứng với 5 id, không dùng làm khóa được.
    - Bỏ chuỗi ngày gốc sau khi đã tách ra năm.
    - Các cột còn lại sẽ quyết ở bước tiền xử lý, tùy câu hỏi. Các cột ID chỉ là mã ngẫu nhiên nên sẽ bỏ trước khi đưa vào mô hình. `track_name`, `track_album_name`, `track_artist` tạm giữ để có thể ghép với dữ liệu ngoài.

## 2.2 Numerical columns

In [43]:
# Để dành convert ở dưới
df['release_date_parsed'] = pd.to_datetime(df['track_album_release_date'], format='mixed')
df['release_year'] = df['release_date_parsed'].dt.year
df['release_year'] = df['release_year'].astype('int32')
df['release_year'].describe()

count    32833.000000
mean      2011.137270
std         11.417451
min       1957.000000
25%       2008.000000
50%       2016.000000
75%       2019.000000
max       2020.000000
Name: release_year, dtype: float64

## 2.3 Categorical columns

## 2.4 Missing data

## 2.5 Relationships

## 2.6 Summary
- Key observations
- Data quality issues
- Preprocessing plan
- Patterns that could lead to questions